<a href="https://colab.research.google.com/github/arman-hossain45/ML_Pipe_Line/blob/main/datathon_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm xgboost catboost optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 10.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
import warnings; warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('/content/dataset.csv')
test = pd.read_csv('/content/test_dataset_comp_7.csv')
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

TARGET = 'Asset_Utilization'
test_timestamp = test['Timestamp'].copy()

In [ ]:
def fe(df):
    df = df.copy()
    if 'Logistics_Delay_Reason' in df.columns:
        df['Logistics_Delay_Reason'] = df['Logistics_Delay_Reason'].fillna('None')
    df['inv_demand_ratio'] = df['Inventory_Level'] / (df['Demand_Forecast'] + 1)
    df['temp_humidity'] = df['Temperature'] * df['Humidity']
    df['txn_per_freq'] = df['User_Transaction_Amount'] / (df['User_Purchase_Frequency'] + 1)
    df['demand_minus_inv'] = df['Demand_Forecast'] - df['Inventory_Level']
    return df

train = fe(train)
test = fe(test)

drop_cols = ['Timestamp', TARGET]
cat_cols = [c for c in ['Asset_ID','Shipment_Status','Traffic_Status','Logistics_Delay_Reason'] if c in train.columns]
num_cols = [c for c in train.columns if c not in cat_cols + drop_cols and c in test.columns]
features = cat_cols + num_cols

for c in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[c].astype(str), test[c].astype(str)], axis=0)
    le.fit(combined)
    train[c] = le.transform(train[c].astype(str))
    test[c] = le.transform(test[c].astype(str))

def kfold_target_encode(train, test, col, target, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    train_enc = np.zeros(len(train))
    global_mean = train[target].mean()
    for tr_idx, val_idx in kf.split(train):
        means = train.iloc[tr_idx].groupby(col)[target].mean()
        train_enc[val_idx] = train.iloc[val_idx][col].map(means).fillna(global_mean)
    test_enc = test[col].map(train.groupby(col)[target].mean()).fillna(global_mean)
    return train_enc, test_enc.values

train['Asset_ID_te'], test['Asset_ID_te'] = kfold_target_encode(train, test, 'Asset_ID', TARGET)
features = features + ['Asset_ID_te']

X = train[features].reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)
X_test = test[features].reset_index(drop=True)

N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

In [ ]:
def lgb_cv_mse(params):
    oof = np.zeros(len(X))
    for tr_idx, val_idx in kf.split(X):
        m = lgb.LGBMRegressor(**params, n_estimators=3000, random_state=42, verbosity=-1)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              categorical_feature=cat_cols, callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[val_idx] = m.predict(X.iloc[val_idx], num_iteration=m.best_iteration_)
    return mean_squared_error(y, oof)

def lgb_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    return lgb_cv_mse(params)

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(lgb_objective, n_trials=30, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print("best lgb mse:", study_lgb.best_value)

  0%|          | 0/30 [00:00<?, ?it/s]

best lgb mse: 135.8653971516288


In [ ]:
def xgb_cv_mse(params):
    oof = np.zeros(len(X))
    for tr_idx, val_idx in kf.split(X):
        m = xgb.XGBRegressor(**params, n_estimators=3000, random_state=42,
                              verbosity=0, early_stopping_rounds=100)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], verbose=False)
        oof[val_idx] = m.predict(X.iloc[val_idx])
    return mean_squared_error(y, oof)

def xgb_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'objective': 'reg:squarederror'
    }
    return xgb_cv_mse(params)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(xgb_objective, n_trials=30, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print("best xgb mse:", study_xgb.best_value)

  0%|          | 0/30 [00:00<?, ?it/s]

best xgb mse: 136.48113859437524


In [ ]:
def cat_cv_mse(params):
    oof = np.zeros(len(X))
    for tr_idx, val_idx in kf.split(X):
        m = CatBoostRegressor(**params, iterations=3000, cat_features=cat_cols,
                               random_state=42, verbose=False, early_stopping_rounds=100)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=(X.iloc[val_idx], y.iloc[val_idx]))
        oof[val_idx] = m.predict(X.iloc[val_idx])
    return mean_squared_error(y, oof)

def cat_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
    }
    return cat_cv_mse(params)

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(cat_objective, n_trials=20, show_progress_bar=True)
best_cat_params = study_cat.best_params
print("best cat mse:", study_cat.best_value)

  0%|          | 0/20 [00:00<?, ?it/s]

best cat mse: 136.83827189050027


In [ ]:
oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cat = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test)); pred_xgb = np.zeros(len(X_test)); pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    m_lgb = lgb.LGBMRegressor(**best_lgb_params, n_estimators=5000, random_state=42, verbosity=-1)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=cat_cols,
              callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict(X_val, num_iteration=m_lgb.best_iteration_)
    pred_lgb += m_lgb.predict(X_test, num_iteration=m_lgb.best_iteration_) / N_SPLITS

    m_xgb = xgb.XGBRegressor(**best_xgb_params, n_estimators=5000, random_state=42,
                              verbosity=0, early_stopping_rounds=150)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict(X_val)
    pred_xgb += m_xgb.predict(X_test) / N_SPLITS

    m_cat = CatBoostRegressor(**best_cat_params, iterations=5000, cat_features=cat_cols,
                               random_state=42, verbose=False, early_stopping_rounds=150)
    m_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cat[val_idx] = m_cat.predict(X_val)
    pred_cat += m_cat.predict(X_test) / N_SPLITS

    print(f"fold {fold} | lgb {mean_squared_error(y_val, oof_lgb[val_idx]):.3f} "
          f"xgb {mean_squared_error(y_val, oof_xgb[val_idx]):.3f} "
          f"cat {mean_squared_error(y_val, oof_cat[val_idx]):.3f}")

fold 0 | lgb 143.524 xgb 144.487 cat 143.187
fold 1 | lgb 153.652 xgb 152.316 cat 153.255
fold 2 | lgb 122.214 xgb 123.668 cat 120.838
fold 3 | lgb 128.986 xgb 128.193 cat 130.237
fold 4 | lgb 130.951 xgb 133.742 cat 136.674


In [ ]:
from scipy.optimize import minimize

def loss(w):
    blend = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat
    return mean_squared_error(y, blend)

res = minimize(loss, x0=[1/3,1/3,1/3], method='Nelder-Mead', bounds=[(0,1)]*3)
w = res.x / res.x.sum()
print("weights:", w)
print("blended OOF MSE:", loss(w))

final_preds = w[0]*pred_lgb + w[1]*pred_xgb + w[2]*pred_cat

weights: [6.72382924e-01 3.27614173e-01 2.90331398e-06]
blended OOF MSE: 135.67165983992658


In [ ]:
submission = pd.DataFrame({
    'Timestamp': test_timestamp,
    'Asset_Utilization': final_preds
})
submission.to_csv('submission.csv', index=False)
submission.head()

,Timestamp,Asset_Utilization
0,2024-08-11 04:43:30,78.736246
1,2024-02-05 00:48:18,79.250598
2,2024-01-09 23:18:24,79.649128
3,2024-03-13 18:47:10,79.967252
4,2024-02-11 10:31:50,80.471807
